## Questão 3 - Carregamento

### Cenário

Após a criação do schema é necessário realizar o carregamento desses dados nesse banco de dados brutos para facilitar as analises posteriores.

### Premissas obrigatórias

- Realize o carregamento de todos os CSVs.
- Utilize obrigatoriamente Python 3.
- Utilize qualquer biblioteca necessária (nativa ou externa) para conexão e carregamento dos dados.
- Não faça tratamentos como: Remoção de nulos ou correção de caracteres especiais
### Tarefa: 
- Escreva um script python para realizar o carregamento de todos os arquivos CSV respeitando o schema criado na questão anterior. 




Nesta etapa, foi desenvolvido o script `load_data.py` para realizar o carregamento de todos os arquivos CSV nas tabelas previamente criadas pelo arquivo `schema.sql`.

A carga foi realizada sem remoção de valores nulos, correção de caracteres especiais ou qualquer transformação dos dados. O objetivo foi preservar os arquivos de origem e respeitar a estrutura definida na questão anterior.

Como a biblioteca `psycopg2` não foi utilizada, o carregamento foi executado por meio do comando `psql`, chamado pelo Python com a biblioteca nativa `subprocess`.

## 3.1 Bibliotecas utilizadas

Foram utilizadas apenas bibliotecas nativas do Python:

- `os`: localização dos arquivos CSV no diretório;
- `re` e `unicodedata`: padronização dos nomes de arquivos para os nomes das tabelas;
- `sys`: encerramento controlado em caso de erro;
- `subprocess`: execução do comando `psql` a partir do Python.

In [1]:
# ============================================================
# Bibliotecas utilizadas no carregamento
# ============================================================

import os
import re
import sys
import subprocess
import unicodedata

## 3.2 Padronização do nome do arquivo

A função abaixo aplica a mesma regra de normalização usada na geração do schema.

Dessa forma, o nome de cada arquivo CSV é associado à tabela correspondente. Por exemplo:

```text
order_items.csv → order_items


### Célula de código 2

```python
# ============================================================
# ETAPA 1 — Padronização do nome do arquivo para a tabela destino
# ============================================================

def normalizar_identificador(nome, prefixo="tabela"):
    """
    Aplica a mesma padronização utilizada na geração do schema.
    """
    # Remove acentos e converte caracteres para ASCII.
    nome = unicodedata.normalize("NFKD", nome)
    nome = nome.encode("ascii", "ignore").decode("ascii")

    # Padroniza o identificador em letras minúsculas.
    nome = nome.strip().lower()

    # Substitui caracteres não permitidos por sublinhado.
    nome = re.sub(r"[^a-z0-9_]+", "_", nome)
    nome = re.sub(r"_+", "_", nome).strip("_")

    # Define um nome padrão se o identificador ficar vazio.
    if not nome:
        nome = prefixo

    # Evita nomes de tabela iniciados por número.
    if nome[0].isdigit():
        nome = f"{prefixo}_{nome}"

    return nome

## 3.3 Localização e carregamento dos arquivos

A função `carregar_csvs` localiza todos os arquivos com extensão `.csv` no diretório informado.

Para cada arquivo, é construído o comando `\copy` do PostgreSQL. Esse comando realiza a leitura do CSV pelo computador local e envia os registros para a tabela correspondente.

A opção `ON_ERROR_STOP=1` interrompe a execução caso ocorra algum erro. Isso evita que a carga seja concluída parcialmente sem sinalização.

In [2]:
# ============================================================
# ETAPA 2 — Localização dos arquivos CSV e carga no PostgreSQL
# ============================================================

def carregar_csvs(diretorio_csv):
    """
    Carrega todos os CSVs nas tabelas PostgreSQL já existentes.
    Nenhum dado é removido ou transformado.
    """

    # Localiza todos os arquivos CSV disponíveis no diretório.
    arquivos_csv = sorted(
        arquivo
        for arquivo in os.listdir(diretorio_csv)
        if arquivo.lower().endswith(".csv")
    )

    # Interrompe a execução se nenhum arquivo for localizado.
    if not arquivos_csv:
        raise FileNotFoundError("Nenhum arquivo CSV foi encontrado.")

    # Processa cada CSV individualmente.
    for arquivo_csv in arquivos_csv:

        # Obtém o caminho absoluto do arquivo para o comando \copy.
        caminho_csv = os.path.abspath(
            os.path.join(diretorio_csv, arquivo_csv)
        ).replace("\\", "/")

        # Associa o nome do arquivo ao nome da tabela de destino.
        nome_tabela = os.path.splitext(arquivo_csv)[0]
        nome_tabela = normalizar_identificador(nome_tabela)

        # Escapa aspas simples que possam existir no caminho do arquivo.
        caminho_sql = caminho_csv.replace("'", "''")

        # Monta o comando de carga sem alteração dos dados de origem.
        comando_copy = (
            f"\\copy {nome_tabela} "
            f"FROM '{caminho_sql}' "
            f"WITH (FORMAT CSV, HEADER TRUE)"
        )

        # Executa o comando psql e interrompe em caso de erro.
        comando_psql = [
            "psql",
            "-v", "ON_ERROR_STOP=1",
            "-c", comando_copy
        ]

        resultado = subprocess.run(
            comando_psql,
            capture_output=True,
            text=True,
            encoding="utf-8"
        )

        # Exibe o erro e encerra o script se a carga falhar.
        if resultado.returncode != 0:
            print(f"Erro ao carregar {arquivo_csv}:")
            print(resultado.stderr)
            sys.exit(1)

        # Confirma o sucesso do carregamento daquele arquivo.
        print(f"Carregado com sucesso: {arquivo_csv}")

    print("Todos os CSVs foram carregados com sucesso.")

## 3.4 Execução e resultado

O script foi executado apontando para o diretório que contém os arquivos CSV.

Cada arquivo foi associado à tabela com o mesmo nome normalizado e carregado por meio do comando `\copy`. Ao final, foram processadas 24 tabelas no PostgreSQL.

A execução abaixo é mantida no notebook para documentar o procedimento. Como a base já foi carregada, ela não deve ser executada novamente no banco atual.

In [ ]:
# ============================================================
# ETAPA 3 — Execução do carregamento
# ============================================================

# Define o diretório que contém os arquivos CSV.
diretorio_csv = "."

# Executa a carga de todos os arquivos localizados.
carregar_csvs(diretorio_csv)

## 3.5 Validação do carregamento

Após o carregamento dos arquivos CSV, precisa ser verificado se as tabelas centrais foram populadas com a quantidade esperada de registros.

Para essa validação, foram consideradas as tabelas `customers`, `orders`, `order_items` e `payments`.

A fórmula utilizada é:

**Fórmula:**

`Total de linhas = Linhas de customers + Linhas de orders + Linhas de order_items + Linhas de payments`

Essa verificação confirma se as principais tabelas de clientes, pedidos, itens e pagamentos foram carregadas no PostgreSQL.

```sql
SELECT
    (
        (SELECT COUNT(*) FROM customers)
        + (SELECT COUNT(*) FROM orders)
        + (SELECT COUNT(*) FROM order_items)
        + (SELECT COUNT(*) FROM payments)
    ) AS total_linhas;
```

Resultado da validação:

```text
customers:   2.000
orders:      48.998
order_items: 147.320
payments:    53.546
--------------------------------
Total:       251.864
```

### SQL - Consulta
```sql
SELECT
    (SELECT COUNT(*) FROM customers) +
    (SELECT COUNT(*) FROM orders) +
    (SELECT COUNT(*) FROM order_items) +
    (SELECT COUNT(*) FROM payments) AS total_linhas;

### Resultado: 251.864

### SQL - Consulta 2

    SELECT COUNT(*) AS quantidade_tabelas
FROM pg_tables
WHERE schemaname = 'public';

### Resultado: 24